<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 20 · 团队负责人如何审核与查看进度

开发者提交之后，团队负责人需要一个可操作的审核入口，也需要知道各项目做到哪里。我们准备一份待审经验和两个子项目，用真实浏览器完成批准，再下载与页面使用同一套投影的报告。

无需模型；需要 Playwright Chromium。先在仓库终端运行 `uv run --group notebooks playwright install chromium`。Notebook 默认使用无界面浏览器，并保存截图；运行到浏览器步骤前也可以打开显示的 URL 自己操作。

路线：项目与候选 → 浏览器审核 → Skill Library → Scope 报告 → JSON/Markdown 摘要一致性。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("20", features=())
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 20", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 给负责人一些值得查看的工作

第一个子项目有交接记录，第二个还没有交接。父子层级用于报告选择，不会自动共享上下文。经验候选明确限定为本篇的教学输入，不夸大验证范围。

In [ ]:
from powercontext.http import (
    CreateArtifactRequest,
    CreateSourceRequest,
    ExperienceProposal,
    ProposeExperienceRequest,
    SourceReference,
)

source = await client.create_source(
    scope_id, CreateSourceRequest(content="本篇教学检查：Decimal('1.999') 不能无提示截断为整数分。")
)
candidate = await client.propose_experience(
    ProposeExperienceRequest(
        scope_id=scope_id,
        proposal=ExperienceProposal(
            situation="amount 精度可能被截断",
            action="转换之前校验精度",
            outcome="本篇只展示审核流程",
            lesson="amount: 精度校验应在整数转换之前",
        ),
        source_refs=[SourceReference(name="content", source_id=source.source_id)],
        artifact_refs=[],
    )
)
children = []
for name in ("已交接的金额任务", "尚无交接的文档任务"):
    children.append(
        await client.create_scope(
            CreateScopeRequest(
                title=name, summary="报告选择实验", parent_scope_id=scope_id, idempotency_key=f"{lab.run_id}:{name}"
            )
        )
    )
child_evidence = await client.create_source(
    children[0].scope_id, CreateSourceRequest(content="已确定金额精度规则；实现尚未完成。")
)
child_citation = {"kind": "source", "source_ref": {"name": "content", "source_id": child_evidence.source_id}}
await client.create_artifact(
    children[0].scope_id,
    CreateArtifactRequest.model_validate({
        "family": "handoff",
        "content": {
            "schema": "powercontext.handoff.v1",
            "objective": "完成 amount 校验",
            "state": [{"text": "已确定精度检查规则；尚需实现。", "citations": [child_citation]}],
            "disposition": "continuable",
            "next_action": {"text": "实现并运行边界检查。", "citations": [child_citation]},
            "omissions": [],
        },
    }),
)
await client.create_artifact(
    scope_id,
    CreateArtifactRequest.model_validate({
        "family": "skill",
        "content": {
            "name": "csv-team-check",
            "description": "Review CSV amount changes.",
            "instructions": "Inspect the actual checks and report their limits.",
            "validation": ["Actual results are visible"],
        },
    }),
)
review_url = f"{lab.base_url}/reviews?scope={scope_id}&candidate={candidate.candidate_id}"
print("审核入口：", review_url)
print("报告入口：", lab.base_url + "/handoff-reports")

## 在真实浏览器中检查并批准

浏览器加载产品自己的 JavaScript，点击批准并确认。完成后，再从公开 API 读回状态，避免把按钮点击成功误认为服务端已经批准。

In [ ]:
from playwright.async_api import async_playwright, expect

from powercontext.http import GetArtifactCandidateRequest

browser_log = []
async with async_playwright() as playwright:
    import os

    browser = await playwright.chromium.launch(
        headless=True, executable_path=os.environ.get("POWERCONTEXT_NOTEBOOK_BROWSER_EXECUTABLE")
    )
    try:
        page = await browser.new_page(viewport={"width": 1440, "height": 1000})
        page.on("pageerror", lambda error: browser_log.append(str(error)))
        await page.goto(review_url)
        await expect(page.locator("#review-candidate-id")).to_have_text(candidate.candidate_id)
        await page.locator("#review-approve").click()
        await page.locator("#review-confirm-approve").click()
        await expect(page.locator("#review-approve-dialog")).not_to_be_visible()
        await page.screenshot(path=str(lab.directory / "review.png"), full_page=True)
        await page.goto(lab.base_url + f"/skills?scope={scope_id}")
        await expect(page.get_by_text("csv-team-check", exact=True).first).to_be_visible()
        await page.screenshot(path=str(lab.directory / "skills.png"), full_page=True)
        await page.goto(lab.base_url + "/handoff-reports")
        await expect(page.locator("#handoff-report")).to_be_visible()
        await page.locator("#scope-select").select_option(f"subtree:{scope_id}")
        await expect(page.locator("#scope-report-rows tr")).to_have_count(3)
        await expect(page.locator("#scope-report-rows").get_by_text("已交接的金额任务", exact=True)).to_be_visible()
        await expect(page.locator("#continuable-count")).to_have_text("1")
        await expect(page.locator("#no-handoff-count")).to_have_text("2")
        async with page.expect_download() as download_info:
            await page.locator("#download-report").click()
        download = await download_info.value
        downloaded_report = lab.directory / "browser-report.md"
        await download.save_as(downloaded_report)
        assert all(child.title in downloaded_report.read_text(encoding="utf-8") for child in children)
        await page.screenshot(path=str(lab.directory / "report.png"), full_page=True)
    finally:
        await browser.close()
approved = await client.get_artifact_candidate(
    GetArtifactCandidateRequest(scope_id=scope_id, candidate_id=candidate.candidate_id)
)
assert approved.status == "approved"
assert not browser_log, browser_log
show({"浏览器批准后的状态": approved.status, "浏览器脚本错误": len(browser_log), "截图目录": str(lab.directory)})

## 冻结一次选择，比较两种报告

API 的 all、exact、subtree 与团队视图对应。这里选择当前 Scope 子树，分别请求 JSON 和 Markdown，检查选择摘要与每次响应自己的内容摘要。每次请求有独立 generated_at 时间，因此两个请求的 report_digest 可以不同；不能把它当作跨请求固定值。

In [ ]:
import httpx

from powercontext.http import GetHandoffReportRequest, GetStatsRequest, ScopeSelection

selection = ScopeSelection.model_validate({"mode": "subtree", "root_scope_id": scope_id})
report = await client.get_handoff_report(GetHandoffReportRequest(selection=selection, format="json"))
markdown = await client.get_handoff_report(GetHandoffReportRequest(selection=selection, format="markdown"))
async with httpx.AsyncClient(base_url=lab.base_url) as http:
    response = await http.post(
        "/v1/handoff-reports/get",
        json={"selection": selection.model_dump(mode="json"), "format": "markdown", "download": True},
    )
    response.raise_for_status()
assert report.selection_digest == response.headers["X-PowerContext-Selection-Digest"]
assert response.headers["X-PowerContext-Report-Digest"] in response.text
assert report.report and report.report["report_digest"] == report.report_digest
assert isinstance(markdown, str) and all(child.title in markdown for child in children)
assert all(child.title in response.text for child in children)
markdown = response.text
(lab.directory / "handoff-report.md").write_text(markdown, encoding="utf-8")
print(markdown)
stats = await client.get_stats(GetStatsRequest(selection=selection))
show(stats)

## 练习与验收

把 selection 换成 exact，仅选第二个子项目，应得到“无交接”的项目状态。不要用父项目的 Handoff 填补它。浏览器截图、报告文件和 API 状态一起构成本篇验收证据。

接下来阅读 [21_observability_recovery.ipynb](21_observability_recovery.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")